***

Preparing Workspace

***

In [ ]:


## Importing packages ---

import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import math
import seaborn as sns

import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.offline import plot
import plotly.subplots as sp
from plotly.subplots import make_subplots
pd.options.display.float_format = '{:.2f}'.format


## Setting file paths ---

user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

if user == 'jfontes':
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

path_config0 = os.path.join(path_git, 'config')

# Set parameters for export file
path_plots = r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring"
print('Export Location: ' + path_plots)


## User defined functions ---

exec(open(os.path.join(path_config0, 'Functions.py')).read())
exec(open(os.path.join(path_git, 'Data', 'Census', 'config', 'census_functions.py')).read())





***

Importing

***

In [ ]:

## Set indicator, estimate, and values to plot ---

indicator_name = 'Pop_6'
estimate = 'ACS5'
value = 'Birth Rate Per 1K People'

# indicator_name = 'Pop_8'
# estimate = 'ACS5'
# value = 'Percentage'


## Importing ---

geography = 'MPO'
file_mpo = f"{indicator_name} {geography} {estimate}.xlsx"
df_mpo = pd.read_excel(os.path.join(path_plots, 'Data', file_mpo), sheet_name = geography)

geography = 'MSA'
file_msa = f"{indicator_name} {geography} {estimate}.xlsx"
df_msa = pd.read_excel(os.path.join(path_plots, 'Data', file_msa), sheet_name = geography)
if 'Population' not in df_msa.columns:
    df_pop = pd.read_excel(os.path.join(path_plots, 'Data', f'Pop_3 {geography} {estimate}.xlsx'), sheet_name = geography)
    df_pop['Race_Ethnicity'] = df_pop['Race_Ethnicity'].map(pop_eth_labels)
    df_msa = df_msa.merge(df_pop[[geography, 'Year', 'Race_Ethnicity', 'Population']], on=[geography, 'Year', 'Race_Ethnicity'], how='left')

geography = 'States'
file_nat = f"{indicator_name} {geography} {estimate}.xlsx"
df_cal = pd.read_excel(os.path.join(path_plots, 'Data', file_nat), sheet_name = geography)

geography = 'National'
file_nat = f"{indicator_name} {geography} {estimate}.xlsx"
df_nat = pd.read_excel(os.path.join(path_plots, 'Data', file_nat), sheet_name = geography)

# display(df_mpo.head(), df_msa.head(), df_cal.head(), df_nat.head())


## Organizing ---

df_msa['Geography'] = 'Peer MSA'
df_mpo['Geography'] = 'SACOG'
df_cal['Geography'] = 'California'
df_nat['Geography'] = 'National'

df_msa = df_msa[~df_msa['Geography'].str.contains('Sac|Yuba')]

wm = lambda x: np.average(x, weights = df_msa.loc[x.index, "Population"])
df_msa = df_msa.groupby(['Year', 'Geography', 'Race_Ethnicity', 'Variable'], as_index = False).agg(Population = ('Population', 'sum'), value = (value, wm), ME = ('Margin of Error', sqrtsumsq))
df_msa = df_msa.sort_values(['Geography', 'Year'], ascending = [True, False])
df_msa = df_msa.rename(columns = {'value':value, 'ME':'Margin of Error'})

df_plot = pd.concat([df_mpo, df_msa, df_cal, df_nat])
df_plot = df_plot[['Geography', 'Year', 'Race_Ethnicity', 'Variable', value, 'Margin of Error']]

df_plot = df_plot.reset_index(drop=True)
print(); print()
display(df_plot.head(), df_plot.tail())

***

Plotting

***

In [ ]:


# Assignment
df = df_plot.copy()
loop_vars          = False
by_race            = False
race_ethnicity     = 'Race_Ethnicity'
variable           = 'Variable'
x                  = 'Year'
y                  = value
color              = 'Geography'
color_discrete_map = color_map_comp
line_dash          = None
markers            = True

if value in ['Percentage', 'Birth Rate Per 1K People']:
    df[value] = round(df[value], 1)

if by_race:
    races = df[race_ethnicity].unique()
    for race in races:
        if loop_vars:
            vars = df[variable].unique()
            for var in vars:
                df2 = df[(df[variable] == var) & (df[race_ethnicity] == race)]
                fig = px.line(df2, x = x, y = y, color = color, color_discrete_map = color_discrete_map, line_dash = line_dash, markers = markers)
                title = f'{indicator_name} {var} {race}'
                fig.update_yaxes(ticksuffix='%')
                # fig.update_yaxes(tickprefix='$', tickformat = ',.0f')
                fig.update_xaxes(dtick=1, range=[df2.Year.min()-0.5, df2.Year.max()+0.5])
                fig.update_traces(hovertemplate="%{y}")
                plot_agol(export=False)
else:
    df = df[df[race_ethnicity] == 'All'].drop(race_ethnicity, axis = 1)
    if loop_vars:
        vars = df[variable].unique()
        for var in vars:
            df2 = df[df[variable] == var]
            fig = px.line(df2, x = x, y = y, color = color, color_discrete_map = color_discrete_map, line_dash = line_dash, markers = markers)
            title = f'{indicator_name} {var}'
            fig.update_yaxes(ticksuffix='%') # dtick=10, range = [0,100]
            # fig.update_yaxes(tickprefix='$', tickformat = ',.0f')
            fig.update_xaxes(dtick=1, range=[df2.Year.min()-0.5, df2.Year.max()+0.5])
            fig.update_traces(hovertemplate="%{y}")
            plot_agol(export=False)
    else:        
        fig = px.line(df, x = x, y = y, color = color, color_discrete_map = color_discrete_map, line_dash = line_dash, markers = markers)
        title = indicator_name
        # fig.update_yaxes(dtick=10, ticksuffix='%', range = [0,100])
        fig.update_xaxes(dtick=1, range=[df.Year.min()-0.5, df.Year.max()+0.5])
        fig.update_traces(hovertemplate="%{y}")
        plot_agol(export=False)



Code graveyard

In [ ]:



# # Assignment
# df = df_plot.copy()
# loop_vars          = True
# by_race            = False
# race_ethnicity     = 'Race_Ethnicity'
# variable           = 'Variable'
# x                  = 'Year'
# y                  = value
# color              = 'Geography'
# color_discrete_map = color_map_comp
# line_dash          = None
# markers            = True

# if value == 'Percentage':
#     df[value] = round(df[value], 1)

# if by_race:
#     df = df[df[race_ethnicity] != 'All']
# else:
#     df = df[df[race_ethnicity] == 'All'].drop(race_ethnicity, axis = 1)

# if loop_vars:
#     vars = df[variable].unique()
#     for var in vars:
#         df2 = df[df[variable] == var]
#         fig = px.line(df2, x = x, y = y, color = color, color_discrete_map = color_discrete_map, line_dash = line_dash, markers = markers)
#         title = f'{indicator_name} {var}'
#         # fig.update_yaxes(dtick=10, ticksuffix='%', range = [0,100])
#         fig.update_xaxes(dtick=1, range=[df.Year.min()-0.5, df.Year.max()+0.5])
#         fig.update_traces(hovertemplate="%{y}")
#         plot_agol(export=False)
# else:
#     fig = px.line(df, x = x, y = y, color = color, color_discrete_map = color_discrete_map, line_dash = line_dash, markers = markers)
#     title = indicator_name
#     # fig.update_yaxes(dtick=10, ticksuffix='%', range = [0,100])
#     fig.update_xaxes(dtick=1, range=[df.Year.min()-0.5, df.Year.max()+0.5])
#     fig.update_traces(hovertemplate="%{y}")
#     plot_agol(export=False)


